# 01 — Khám phá dữ liệu (EDA)

Phụ trách: **Thành viên 1**. EDA trên tập phát hiện biển báo (thống kê box, không crop từng biển).

## 0) Bootstrap (local / Kaggle / Colab)

- **Local**: notebook nằm trong repo nên dùng luôn repo hiện tại (không đụng vào git của bạn).
- **Kaggle / Colab**: clone repo về `/kaggle/working`, lấy đúng nhánh rồi cài thư viện. Trên Kaggle bật **Internet: On**.
- Repo private: xem `README.md` mục 12.8 (token lưu trong Kaggle Secrets).

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "TODO_URL"             # TODO: URL repo GitHub của nhóm
REPO_DIR = "TSD-Voice-Alert"
REPO_BRANCH = "main"              # Thành viên 1: data-yolo · Thành viên 2: faster-rcnn (khi nhánh đã có trên origin)

ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB = "google.colab" in sys.modules

def sh(cmd: str, check: bool = True) -> int:
    print(">>", cmd)
    return subprocess.run(cmd, shell=True, check=check).returncode

# 1. Tìm repo đang chứa notebook này (local)
local_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "ts_alert").is_dir() and (p / "configs" / "shared.yaml").exists()), None)

if local_root is not None:
    repo_path = local_root
    print("Dùng repo hiện tại:", repo_path)
else:
    # 2. Chưa có repo: clone rồi lấy đúng nhánh (chỉ làm trong môi trường cloud)
    assert REPO_URL != "TODO_URL", "Điền REPO_URL ở ô này trước khi chạy trên Kaggle/Colab."
    base = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
    repo_path = base / REPO_DIR
    if not (repo_path / ".git").exists():
        sh(f'git clone --branch {REPO_BRANCH} {REPO_URL} "{repo_path}"')
    git = f'git -C "{repo_path}"'
    sh(f"{git} fetch origin {REPO_BRANCH}")
    sh(f"{git} checkout -B {REPO_BRANCH} origin/{REPO_BRANCH}")
    sh(f"{git} log --oneline -1")

os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
print("cwd:", Path.cwd())

# Cài thư viện: chỉ trên Kaggle/Colab. Local thì dùng môi trường đã cài bằng requirements.txt.
if ON_KAGGLE or ON_COLAB:
    sh('pip install -q -e .')

import ts_alert
print("Bootstrap OK.")

In [ ]:
RUNTIME = "configs/runtime/kaggle.yaml" if ON_KAGGLE else "configs/runtime/local.yaml"
print("Runtime:", RUNTIME)

## 1) Cấu hình và đường dẫn dữ liệu

Trên Kaggle: Add Input → `maitam/vietnamese-traffic-signs`, kiểm tra tên thư mục trong `configs/runtime/kaggle.yaml` (`paths.raw_dir`).

In [ ]:
from ts_alert.config.resolve import resolve_config, resolve_path

cfg = resolve_config(None, RUNTIME)
DATA_ROOT = resolve_path(cfg["paths"]["raw_dir"])
print("DATA_ROOT =", DATA_ROOT, "| tồn tại:", DATA_ROOT.exists())

## 2) Chạy các hàm EDA

Các hàm trong `analysis/eda.py` phân tích toàn diện và ghi hình trực tiếp vào `analysis/figures/raw/` (dữ liệu thô) và `analysis/figures/processed/` (dữ liệu đã tiền xử lý).

Tương đương dòng lệnh CLI:
- Phân tích tập thô: `python -m analysis.eda --data-root data/raw`
- Phân tích tập processed: `python -m analysis.eda --processed-root data/processed`

In [ ]:
from analysis import eda

# Chạy phân tích toàn diện tập thô (Raw EDA) -> sinh trọn bộ 8 biểu đồ trong analysis/figures/raw/
eda.analyze_raw_dataset(DATA_ROOT)

# Đồng thời vẫn hỗ trợ các hàm độc lập theo quy ước (lưu trực tiếp vào figures/raw/):
# eda.class_distribution(DATA_ROOT)     # số box theo từng lớp -> figures/raw/01_class_distribution.png
# eda.box_size_stats(DATA_ROOT)         # kích thước box -> figures/raw/03_box_dimensions.png
# eda.center_heatmap(DATA_ROOT)         # vị trí tâm box -> figures/raw/04_box_center_heatmap.png

## 3) Trực quan hóa các biểu đồ phân tích (Raw & Processed)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# 1. Hiển thị toàn bộ các biểu đồ của tập dữ liệu thô (Raw)
raw_fig_dir = eda.FIG_DIR / "raw"
if raw_fig_dir.is_dir():
    raw_figs = sorted(raw_fig_dir.glob("*.png"))
    print(f"=== Tập dữ liệu thô (Raw EDA): {len(raw_figs)} hình ===")
    for f in raw_figs:
        plt.figure(figsize=(9, 5))
        plt.imshow(mpimg.imread(f))
        plt.title(f"[Raw] {f.name}", fontsize=11, fontweight="bold")
        plt.axis("off")
        plt.show()

# 2. Hiển thị các biểu đồ của tập dữ liệu sau tiền xử lý (Processed) nếu đã chạy preprocess
proc_fig_dir = eda.FIG_DIR / "processed"
if proc_fig_dir.is_dir():
    proc_figs = sorted(proc_fig_dir.glob("*.png"))
    if proc_figs:
        print(f"\n=== Tập dữ liệu sau tiền xử lý (Processed EDA): {len(proc_figs)} hình ===")
        for f in proc_figs:
            plt.figure(figsize=(9, 5))
            plt.imshow(mpimg.imread(f))
            plt.title(f"[Processed] {f.name}", fontsize=11, fontweight="bold")
            plt.axis("off")
            plt.show()


## 4) Nhận xét (Thành viên 1 — Phân tích Khám phá Dữ liệu EDA)

1. **Quy mô và Độ mất cân bằng lớp**:
   - Dataset thô gồm 3.216 ảnh với 8.608 bounding box trải trên 52 lớp biển báo chuẩn QCVN 41:2019/BGTVT.
   - Phân bố nhãn có hiện tượng lệch đuôi dài (Long-tail) rất nghiêm trọng: Lớp nhiều nhất là `P.130 (Cấm dừng và đỗ xe)` với 985 box, trong khi các lớp hiếm như `Camera`, `S.505a`, `B.8a` chỉ có dưới 15-20 box.

2. **Đặc tính Vật thể nhỏ (Small Object Detection)**:
   - Hơn 75% biển báo có kích thước diện tích dưới 5% khung hình (nhóm Small và Medium theo thang đo MS-COCO).
   - Có **811 bounding box nhỏ hơn 8px** trên canvas 640x640. Các box này là nhiễu biên cực mờ, bắt buộc phải loại bỏ trong bước tiền xử lý để tránh gây hại cho quá trình học đặc trưng của cả YOLO và Faster R-CNN.

3. **Phân bố không gian (Spatial Heatmap) & Gom cụm bối cảnh**:
   - Heatmap tâm box tập trung lệch mạnh về nửa bên phải và phần trên của ảnh, phản ánh đúng thực tế giao thông Việt Nam (biển báo đặt ở lề đường bên phải hoặc gắn trên giá long môn).
   - Bằng dHash 1024-bit (Hamming <= 380), phát hiện 3.216 ảnh gốc thực chất thuộc về **314 cụm bối cảnh liên tiếp**. Nếu chia ngẫu nhiên (random split) ở mức ảnh sẽ gây rò rỉ dữ liệu trầm trọng giữa train và val/test.
